# Voltage sequence lab (sticky DC)

A 10-minute introduction to sticky DC control with `VoltageGate`, `GateSet`, and `VoltageSequence`.

You will:

1. Create sticky **`VoltageGate`** channels with a `"half_max_square"` pulse.
2. Group them in a **`GateSet`** and add a named point.
3. Inside `program()`, call `new_sequence()` then `step_to_point`.
4. See **`keep_levels=True`**: omitted gates keep their last value.

**Runs without QM hardware** — no `machine.connect()`. The cells only build a QUA program in memory.

Deep guide (virtualization, compensation, timing): [voltage_sequence/README.md](../quam_builder/architecture/quantum_dots/voltage_sequence/README.md).

## 1. Sticky channels and a GateSet

Channels must be **sticky**. Each needs a `"half_max_square"` operation. Dictionary keys must match the names you pass to `add_point` and `step_to_voltages`.

In [ ]:
from quam.components import StickyChannelAddon, pulses
from quam_builder.architecture.quantum_dots.components import VoltageGate, GateSet
from qm import qua

channel_p1 = VoltageGate(
    opx_output=("con1", 1),
    sticky=StickyChannelAddon(duration=1_000, digital=False),
    operations={"half_max_square": pulses.SquarePulse(amplitude=0.25, length=1000)},
)
channel_p2 = VoltageGate(
    opx_output=("con1", 2),
    sticky=StickyChannelAddon(duration=1_000, digital=False),
    operations={"half_max_square": pulses.SquarePulse(amplitude=0.25, length=1000)},
)

gate_set = GateSet(
    id="dot_plungers",
    channels={"channel_p1": channel_p1, "channel_p2": channel_p2},
)
print("GateSet channels:", list(gate_set.channels))

## 2. Named point

`add_point` stores a `VoltageTuningPoint` on the gate set. Use the **same** channel names as in the `GateSet`.

In [ ]:
gate_set.add_point(
    name="idle",
    voltages={"channel_p1": 0.1, "channel_p2": -0.05},
    duration=1000,
)
print("Named points:", list(gate_set.get_macros()))

## 3. Sequence in a QUA program

`new_sequence()` must run **inside** `program()`. Default is `keep_levels=True`: a later `step_to_voltages` that names only `channel_p1` leaves `channel_p2` at its last value. Pass an explicit `0.0` (or `keep_levels=False`) to drive an omitted gate to 0 V.

This cell only compiles the program; it does not play pulses on hardware.

In [ ]:
with qua.program() as prog:
    seq = gate_set.new_sequence()  # keep_levels=True
    seq.step_to_point("idle")
    seq.step_to_voltages({"channel_p1": 0.2}, duration=1000)  # channel_p2 held at -0.05
    seq.step_to_voltages({"channel_p2": 0.0}, duration=500)  # channel_p1 held at 0.2

print("Built QUA program successfully.")

## Next

- Virtual gates, compensation pulses, rectangular matrices, detuning: [voltage_sequence/README.md](../quam_builder/architecture/quantum_dots/voltage_sequence/README.md) (start at [§8](../quam_builder/architecture/quantum_dots/voltage_sequence/README.md#8-full-end-to-end-example)).
- Macros on a full machine: [macro_customization.ipynb](macro_customization.ipynb).